In [1]:
# =====================================================================
# ĐOẠN CODE R ÉP KAGGLE CHẠY MÃ NGUỒN PYTHON (DÙNG CHO NOTEBOOK R)
# =====================================================================
library(reticulate)

# Viết mã nguồn Python vào một file tạm script.py
cat("
import os
import re
import cv2
import random
import shutil
import time
import sys
from pathlib import Path

start_total_time = time.time()

IP102_BASE = Path('/kaggle/input/datasets/rtlmhjbn/ip02-dataset')
UNIFIED_BASE = Path('/kaggle/input/datasets/synyyy/plantdataset')

BASE_OUT = Path('/kaggle/working/unified_dataset')
IMG_OUT = BASE_OUT / 'images'
LBL_OUT = BASE_OUT / 'labels'
os.makedirs(IMG_OUT, exist_ok=True)
os.makedirs(LBL_OUT, exist_ok=True)

START_UNIFIED_ID = 102
unified_class_mapping = {}
current_unified_id = START_UNIFIED_ID

print('==================================================', flush=True)
print('⏳ BƯỚC 1: ĐANG ĐỌC FILE DANH SÁCH CLASS IP102...', flush=True)
try:
    with open('/kaggle/input/datasets/rtlmhjbn/ip02-dataset/classes.txt', 'r') as f:
        ip102_class_names = [line.strip().lower() for line in f.readlines() if line.strip()]
    print(f'✅ Đã tải thành công {len(ip102_class_names)} class của bộ IP102.', flush=True)
except Exception as e:
    print(f'❌ LỖI: Không tìm thấy file classes.txt: {str(e)}', flush=True)
    sys.exit()

print('\\n==================================================', flush=True)
print('⏳ BƯỚC 2: ĐANG ĐỒNG BỘ BỘ DỮ LIỆU CÔN TRÙNG IP102...', flush=True)
start_time = time.time()
ip102_pairs = []
count_ip102_processed = 0

all_ip102_images = list(IP102_BASE.rglob('*.*'))
total_ip102 = len(all_ip102_images)
print(f'🔍 Tìm thấy tổng cộng {total_ip102} file trong thư mục IP102.', flush=True)

for img_path in all_ip102_images:
    if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png']:
        try:
            folder_name = img_path.parent.name
            class_id = int(folder_name) - 1 
        except ValueError:
            continue 
            
        if 0 <= class_id < 102:
            lbl_name = f'ip102_{img_path.stem}.txt'
            new_img_name = f'ip102_{img_path.name}'
            
            with open(LBL_OUT / lbl_name, 'w') as f:
                f.write(f'{class_id} 0.5 0.5 1.0 1.0\\n')
                
            shutil.copy(img_path, IMG_OUT / new_img_name)
            ip102_pairs.append(str(IMG_OUT / new_img_name))
            count_ip102_processed += 1
            
            if count_ip102_processed % 5000 == 0:
                print(f'   [Tiến độ IP102]: Đã gộp {count_ip102_processed}/{total_ip102} ảnh ({count_ip102_processed/total_ip102*100:.1f}%)', flush=True)

print(f'✅ HOÀN THÀNH IP102: Đã xử lý {len(ip102_pairs)} ảnh. Thời gian: {time.time() - start_time:.2f} giây.', flush=True)

print('\\n==================================================', flush=True)
print('⏳ BƯỚC 3: ĐANG ĐỒNG BỘ BỘ DỮ LIỆU BỆNH LÁ UNIFIED...', flush=True)
start_time = time.time()
unified_pairs = []
count_unified_processed = 0

all_unified_images = list(UNIFIED_BASE.rglob('*.*'))
total_unified = len(all_unified_images)
print(f'🔍 Tìm thấy tổng cộng {total_unified} file trong thư mục Unified.', flush=True)

for img_path in all_unified_images:
    if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png']:
        disease_folder = img_path.parent.name
        plant_folder = img_path.parent.parent.name
        metadata_name = f'{plant_folder}_{disease_folder}'.lower().strip()
        
        if metadata_name not in unified_class_mapping:
            unified_class_mapping[metadata_name] = current_unified_id
            current_unified_id += 1
            
        class_id = unified_class_mapping[metadata_name]
        new_img_name = f'unifi_{plant_folder}_{disease_folder}_{img_path.name}'
        lbl_name = f'unifi_{plant_folder}_{disease_folder}_{img_path.stem}.txt'
        
        with open(LBL_OUT / lbl_name, 'w') as f:
            f.write(f'{class_id} 0.5 0.5 1.0 1.0\\n')
            
        shutil.copy(img_path, IMG_OUT / new_img_name)
        unified_pairs.append(str(IMG_OUT / new_img_name))
        count_unified_processed += 1
        
        if count_unified_processed % 5000 == 0:
            print(f'   [Tiến độ Unified]: Đã xử lý {count_unified_processed}/{total_unified} ảnh ({count_unified_processed/total_unified*100:.1f}%)', flush=True)

print(f'✅ HOÀN THÀNH UNIFIED: Phát hiện {len(unified_class_mapping)} lớp bệnh. Đã tạo hộp {len(unified_pairs)} ảnh. Thời gian: {time.time() - start_time:.2f} giây.', flush=True)

print('\\n==================================================', flush=True)
print('⏳ BƯỚC 4: ĐANG TIẾN HÀNH PHÂN PHỐI ĐỀU DỮ LIỆU THÀNH 3 PHẦN...', flush=True)
start_time = time.time()
random.seed(42)
random.shuffle(ip102_pairs)
random.shuffle(unified_pairs)

def split_into_three(lst):
    size = len(lst) // 3
    return [lst[0:size], lst[size:2*size], lst[2*size:]]

ip102_splits = split_into_three(ip102_pairs)
unified_splits = split_into_three(unified_pairs)

SPLIT_TXT_DIR = Path('/kaggle/working/split_metadata')
os.makedirs(SPLIT_TXT_DIR, exist_ok=True)

for i in range(3):
    account_data = ip102_splits[i] + unified_splits[i]
    random.shuffle(account_data) 
    
    txt_file_path = SPLIT_TXT_DIR / f'train_account_{i+1}.txt'
    with open(txt_file_path, 'w') as f:
        f.write('\\n'.join(account_data))
    
    print(f'📦 [Tài khoản Kaggle {i+1}]:', flush=True)
    print(f'   - Số lượng ảnh Côn trùng (IP102): {len(ip102_splits[i])} ảnh', flush=True)
    print(f'   - Số lượng ảnh Bệnh lá (Unified): {len(unified_splits[i])} ảnh', flush=True)
    print(f'   - Tổng cộng: {len(account_data)} ảnh -> Đã xuất train_account_{i+1}.txt', flush=True)

print(f'✅ HOÀN THÀNH PHÂN CHIA HỆ THỐNG. Thời gian: {time.time() - start_time:.2f} giây.', flush=True)

print('\\n==================================================', flush=True)
print('⏳ BƯỚC 5: ĐANG XUẤT FILE CẤU HÌNH DATA.YAML VỚI METADATA ĐỒNG BỘ CẢ 2 BỘ...', flush=True)
yaml_path = BASE_OUT / 'data.yaml'
total_classes = len(ip102_class_names) + len(unified_class_mapping)

with open(yaml_path, 'w') as f:
    f.write('# File cấu hình đồng bộ hóa toàn diện hệ thống\\n')
    f.write(f'nc: {total_classes}\\n')
    f.write('names:\\n')
    
    for idx, name in enumerate(ip102_class_names):
        f.write(f'  {idx}: \"[IP102_Insect] {name}\"\\n')
        
    for name, idx in unified_class_mapping.items():
        f.write(f'  {idx}: \"[Unified_Disease] {name}\"\\n')

print(f'✅ FILE CẤU HÌNH SẴN SÀNG: Tổng số class đã đồng bộ metadata là {total_classes}.', flush=True)
print(f'📍 File yaml lưu tại: {yaml_path}', flush=True)
print('==================================================', flush=True)
print(f'🎉 TẤT CẢ HOÀN THÀNH ĐỒNG BỘ DỮ LIỆU! Tổng thời gian chạy: {(time.time() - start_total_time)/60:.2f} phút.', flush=True)
print('==================================================', flush=True)
", file = "script.py")

# Sử dụng lệnh hệ thống của R để gọi Python chạy file script.py vừa tạo
system("python3 script.py")
